# Notebook 05 — Power BI Data Preparation

**Goal:** Build the final denormalized flat table (`data/powerbi/powerbi_export.csv`) that serves as the single data source for the 5-page Power BI dashboard.

**What this notebook does:**
1. Joins all 8 tables into a single flat table (one row per order line item)
2. Adds all computed/derived columns: region, delivery days, cohort month, RFM segment
3. Validates the output (100,000+ rows, no critical nulls)
4. Exports to both CSV and XLSX

**Pre-requisites:** Notebooks 01, 03, and 04 must have been run (DB needs `rfm_segments` and `customer_cohorts` tables).

---

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

from src.db_setup import get_engine, DB_PATH
from src.etl import (
    build_master_flat_table,
    attach_rfm_segments,
    attach_cohort_month,
    export_powerbi_csv,
    STATE_REGION_MAP,
)

sns.set_theme(style='whitegrid')
engine = get_engine()
pd.set_option('display.float_format', '{:,.2f}'.format)
print(f'Connected: {DB_PATH}')

## Step 1: Build the Master Flat Table

The SQL join chain:
```
orders
  JOIN customers
  JOIN order_items
    JOIN products
    JOIN category_translation
    JOIN sellers
  LEFT JOIN payments (aggregated per order)
  LEFT JOIN reviews  (aggregated per order)
```

Result: one row per `(order_id, order_item_id)` — 112,000+ rows.

In [ ]:
print('Building master flat table...')
df = build_master_flat_table(engine)
print(f'\nShape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## Step 2: Attach RFM Segment Labels

In [ ]:
df = attach_rfm_segments(df, engine)
print('RFM segment distribution in flat table:')
print(df['rfm_segment'].value_counts().to_string())

## Step 3: Attach Cohort Month

In [ ]:
df = attach_cohort_month(df, engine)
print(f'Cohort month populated for {df["customer_cohort_month"].notna().sum():,} rows')
print(f'Sample cohort months: {sorted(df["customer_cohort_month"].dropna().unique())[:5]}')

## Step 4: Data Quality Check

In [ ]:
print('Null counts for key columns:')
key_cols = [
    'order_id', 'order_date', 'order_year', 'customer_id',
    'customer_state', 'customer_region', 'product_category_english',
    'price', 'total_item_value', 'rfm_segment', 'customer_cohort_month'
]
null_counts = df[[c for c in key_cols if c in df.columns]].isnull().sum()
print(null_counts.to_string())

print(f'\nTotal rows: {len(df):,}')
print(f'Date range: {df["order_date"].min().date()} → {df["order_date"].max().date()}')
print(f'Unique orders: {df["order_id"].nunique():,}')
print(f'Unique customers: {df["customer_id"].nunique():,}')
print(f'Order years: {sorted(df["order_year"].dropna().unique().astype(int).tolist())}')

## Step 5: Summary Statistics for Power BI Dashboard KPIs

In [ ]:
total_revenue = df['total_item_value'].sum()
total_orders  = df['order_id'].nunique()
avg_order_val = df.groupby('order_id')['total_item_value'].sum().mean()
total_customers = df['customer_id'].nunique()

print('=== Power BI Executive KPIs ===')
print(f'Total Revenue:       BRL {total_revenue:>15,.2f}')
print(f'Total Orders:              {total_orders:>12,}')
print(f'Avg Order Value:     BRL {avg_order_val:>15,.2f}')
print(f'Total Customers:           {total_customers:>12,}')
print(f'Product Categories:        {df["product_category_english"].nunique():>12,}')
print(f'States covered:            {df["customer_state"].nunique():>12,}')

In [ ]:
# Compute YoY total revenue growth (2017 → 2018)
rev_by_year = df.groupby('order_year')['total_item_value'].sum()
print('Revenue by year:')
print(rev_by_year.to_string())

if 2017 in rev_by_year.index and 2018 in rev_by_year.index:
    yoy_growth = (rev_by_year[2018] - rev_by_year[2017]) / rev_by_year[2017] * 100
    print(f'\nYoY total revenue growth 2017→2018: {yoy_growth:.1f}%')
else:
    print('\nNote: Cannot compute YoY — check order_year values.')

## Step 6: Delivery Days Distribution

In [ ]:
delivery = df['delivery_days'].dropna()
delivery_positive = delivery[delivery > 0]

print(f'Orders with delivery data: {len(delivery_positive):,}')
print(f'Avg delivery time: {delivery_positive.mean():.1f} days')
print(f'Median delivery:   {delivery_positive.median():.1f} days')
print(f'95th percentile:   {delivery_positive.quantile(0.95):.1f} days')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(delivery_positive[delivery_positive < 60], bins=50, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(delivery_positive.mean(), color='red', linestyle='--', label=f'Mean: {delivery_positive.mean():.1f}d')
ax.axvline(delivery_positive.median(), color='orange', linestyle='--', label=f'Median: {delivery_positive.median():.1f}d')
ax.set_title('Delivery Time Distribution (days)', fontsize=12)
ax.set_xlabel('Delivery Days')
ax.set_ylabel('Order Count')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/delivery_days_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Review Score Distribution

In [ ]:
rev_scores = df['review_score'].dropna()
print('Review score distribution:')
print(rev_scores.value_counts().sort_index().to_string())
print(f'Average review score: {rev_scores.mean():.2f}')

## Step 8: Export to CSV and XLSX

In [ ]:
csv_path = export_powerbi_csv(df)
print(f'\nExport complete. File: {csv_path}')

## Step 9: Validate Export

In [ ]:
verification = pd.read_csv(csv_path, nrows=5)
print(f'CSV columns ({len(verification.columns)}): {list(verification.columns)}')

full_check = pd.read_csv(csv_path)
print(f'Total rows in export: {len(full_check):,}')
print(f'rfm_segment populated: {(full_check["rfm_segment"] != "Unscored").sum():,} rows')
print(f'cohort_month populated: {(full_check["customer_cohort_month"] != "Unknown").sum():,} rows')

assert len(full_check) >= 100_000, f'Expected 100k+ rows, got {len(full_check):,}'
print('\nValidation PASSED: 100,000+ rows confirmed.')

## Step 10: Update Final KPI Summary

In [ ]:
import json
kpi_path = project_root / 'reports' / 'kpi_summary.json'
existing = json.loads(kpi_path.read_text()) if kpi_path.exists() else {}

existing['executive_kpis'] = {
    'total_revenue_brl': round(float(total_revenue), 2),
    'total_orders': int(total_orders),
    'avg_order_value_brl': round(float(avg_order_val), 2),
    'total_customers': int(total_customers),
    'product_categories': int(df['product_category_english'].nunique()),
    'states_covered': int(df['customer_state'].nunique()),
    'powerbi_export_rows': int(len(full_check)),
}

if 2017 in rev_by_year.index and 2018 in rev_by_year.index:
    existing['executive_kpis']['yoy_total_growth_2017_2018_pct'] = round(float(yoy_growth), 1)

kpi_path.write_text(json.dumps(existing, indent=2, default=str))
print('KPI summary updated.')
print(json.dumps(existing.get('executive_kpis', {}), indent=2))

## Summary

The Power BI export table is ready at `data/powerbi/powerbi_export.csv` with:

| Column | Description |
|---|---|
| `order_id` … `order_quarter` | Order timing dimensions |
| `customer_state`, `customer_region` | Geographic dimensions |
| `product_category_english` | Category (English) |
| `price`, `freight_value`, `total_item_value` | Revenue measures |
| `payment_type`, `payment_installments` | Payment attributes |
| `review_score` | Satisfaction metric |
| `delivery_days` | Logistics KPI |
| `rfm_segment` | Customer segment |
| `customer_cohort_month` | First-purchase cohort |
| `is_high_value_customer` | Top-20% flag |

**Next step:** Open Power BI Desktop and follow `powerbi/README.md` to build the 5-page dashboard.